# Bidirectional BLAST - manual QC notebook

Companion notebook to `src/run_bidirectional_blast.py`, for manually inspecting the reciprocal-BLAST pipeline result by result.

**No DIAMOND commands are run here.** Every step below loads the intermediary tables that `run_bidirectional_blast.py` already wrote to disk (`self_id/*.tsv`, `forward/*.tsv`, `reverse_*.tsv`, ...) and replays the same pandas logic the script uses (Steps 0-4), so you can inspect the DataFrame at any stage. Cells are ordered to mirror the script section by section.

Cells are **not** executed - run them yourself.

Set `OUTPUT_DIR` below to the run you want to inspect. It currently points at `results/diamond_rbh_actinos_my_proteins`, which is the run that actually has output on disk (the `config/streptomycetae.yaml` currently in the repo points at `results/diamond_reverseBLAST_streptomycetae`, which doesn't exist yet - update `OUTPUT_DIR` if you re-run with that config).

In [1]:
from pathlib import Path

import pandas as pd
import yaml

## Paths & constants

Copied straight from `run_bidirectional_blast.py` so the column names line up exactly with the files on disk.

In [2]:
PROJECT_DIR = Path("/vol/local/calarass/Projects/ara_comp_genomics")
CONFIG_PATH = PROJECT_DIR / "config" / "streptomycetae.yaml"

# Points at the run that currently has output on disk. Swap this for another
# output_dir (e.g. from a fresh config run) to inspect a different run.
OUTPUT_DIR = PROJECT_DIR / "results" / "diamond_rbh_actinos_my_proteins"

FWD_DIR = OUTPUT_DIR / "forward"
SELF_ID_DIR = OUTPUT_DIR / "self_id"

with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

FASTA_DIR = Path(cfg["fasta_dir"])
REFERENCE_PROTEOMES = cfg["reference_proteomes"]
SELF_ID_MIN_PIDENT = cfg["self_id_min_pident"]
GENOME_SEP = cfg.get("genome_sep")

# Output schema for each DIAMOND call - must match FWD_COLS/REV_COLS/SELF_ID_COLS
# in run_bidirectional_blast.py, since these files have no header row.
FWD_COLS = ["qseqid", "sseqid", "pident", "length", "qlen", "slen",
            "qstart", "qend", "sstart", "send", "evalue", "bitscore",
            "full_sseq"]
REV_COLS = ["qseqid", "sseqid", "pident", "evalue", "bitscore"]
SELF_ID_COLS = ["qseqid", "sseqid", "pident", "bitscore"]


def reference_group(query_file):
    upper = query_file.upper()
    for group in REFERENCE_PROTEOMES:
        if upper.startswith(group):
            return group
    raise ValueError(f"Cannot determine reference organism for query id: {query_file}")


def bare_id(raw_id):
    parts = raw_id.split("|")
    return parts[1] if len(parts) > 1 else raw_id


def read_tsv(path, names):
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame(columns=names)
    return pd.read_csv(path, sep="\t", header=None, names=names)


sorted(p.name for p in OUTPUT_DIR.glob("*"))

['all_forward_hits_flagged.tsv',
 'forward',
 'forward_hits_SCO.faa',
 'forward_hits_VNZ.faa',
 'possible_paralogs.tsv',
 'reciprocal_hits.tsv',
 'reference_proteome_SCO.dmnd',
 'reverse_SCO.tsv',
 'reverse_VNZ.tsv',
 'reverse_hits_combined.tsv',
 'self_id',
 'summary_per_query.tsv']

## Step 0 - self-id tables

Loads the already-computed `self_id/*_self.tsv` tables (query vs. its own reference proteome) instead of re-running DIAMOND, and reconstructs `true_ref_ids` with the same best-hit-by-bitscore + pident-threshold logic as `find_self_id()`.

In [9]:
self_id_tables = {}
true_ref_ids = {}

for fasta_file in sorted(FASTA_DIR.glob("*.fasta")):
    stem = fasta_file.stem
    hits = read_tsv(SELF_ID_DIR / f"{stem}_self.tsv", SELF_ID_COLS)    
    self_id_tables[stem] = hits
    #print(hits.head())


    best = hits.sort_values("bitscore", ascending=False).iloc[0]
    assert best.pident >= SELF_ID_MIN_PIDENT, (
        f"{stem}: best self-hit is only {best.pident:.1f}% identical "
        f"(< {SELF_ID_MIN_PIDENT}%)"
    )
    true_ref_ids[stem] = bare_id(best.sseqid)

true_ref_ids

{'SCO2401': 'SCO2401',
 'SCO2402': 'SCO2402',
 'SCO2403': 'SCO2403',
 'SCO2407': 'SCO2407',
 'SCO2439': 'SCO2439',
 'SCO2440': 'SCO2440',
 'vnz_33170': 'vnz_RS33505',
 'vnz_33175': 'vnz_RS33510',
 'vnz_33180': 'vnz_RS33515',
 'vnz_33185': 'vnz_RS33520'}

In [10]:
# Inspect one query's self-id hits manually, e.g.:
self_id_tables["SCO2401"]

,qseqid,sseqid,pident,bitscore
0,SCO2401,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,99.7,791


## Step 1 - forward hits

Loads the already-computed `forward/*_forward.tsv` tables instead of re-running DIAMOND against the combined genome database.

In [14]:
fwd_tables = []
for fasta_file in sorted(FASTA_DIR.glob("*.fasta")):
    print(f"Reading forward hits for {fasta_file.name}")
    stem = fasta_file.stem
    #print(FWD_DIR / f"{stem}_forward.tsv")
    df = read_tsv(FWD_DIR / f"{stem}_forward.tsv", FWD_COLS)
    df["query_file"] = stem
    fwd_tables.append(df)

fwd_hits = pd.concat(fwd_tables, ignore_index=True)
fwd_hits["ref_group"] = fwd_hits.query_file.map(reference_group)
fwd_hits["true_ref_id"] = fwd_hits.query_file.map(true_ref_ids)

print(f"{len(fwd_hits)} forward hits, {fwd_hits.sseqid.nunique()} unique subject proteins")
fwd_hits.head()

Reading forward hits for SCO2401.fasta
Reading forward hits for SCO2402.fasta
Reading forward hits for SCO2403.fasta
Reading forward hits for SCO2407.fasta
Reading forward hits for SCO2439.fasta
Reading forward hits for SCO2440.fasta
Reading forward hits for vnz_33170.fasta
Reading forward hits for vnz_33175.fasta
Reading forward hits for vnz_33180.fasta
Reading forward hits for vnz_33185.fasta
848 forward hits, 848 unique subject proteins


,qseqid,sseqid,pident,length,qlen,slen,qstart,qend,sstart,send,evalue,bitscore,full_sseq,query_file,ref_group,true_ref_id
0,SCO2401,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,99.7,388,388,388,1,388,1,388,1.170000e-291,791,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401
1,SCO2401,Streptomyces_violaceoruber|R2E43_RS26250|mande...,99.7,388,388,388,1,388,1,388,1.170000e-291,791,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401
2,SCO2401,Streptomyces_anthocyanicus|OIE72_RS26120|mande...,99.5,388,388,388,1,388,1,388,3.360000e-291,790,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401
3,SCO2401,Streptomyces_rubrogriseus|Sru02f_RS06985|mande...,99.0,388,388,388,1,388,1,388,2.760000e-290,788,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401
4,SCO2401,Streptomyces_coelicoflavus|OHA99_RS25815|mande...,98.7,388,388,388,1,388,1,388,1.600000e-289,786,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401


In [15]:
# Quick per-query overview
fwd_hits.groupby("query_file").size().sort_values(ascending=False)

query_file
SCO2440      133
SCO2401      130
SCO2403      130
SCO2439      125
SCO2407       95
SCO2402       94
vnz_33170     43
vnz_33175     36
vnz_33180     35
vnz_33185     27
dtype: int64

## Step 2 + 3 - reverse hits

Loads the already-computed `reverse_<group>.tsv` tables instead of writing `forward_hits_<group>.faa` and re-running DIAMOND against each reference organism's own proteome.

Also cross-checks the reconstructed table against `reverse_hits_combined.tsv`, which the script already wrote from the same data - the two should match exactly.

In [16]:
rev_tables = []
for group in REFERENCE_PROTEOMES:
    rev_group = read_tsv(OUTPUT_DIR / f"reverse_{group}.tsv", REV_COLS)
    if rev_group.empty:
        continue
    rev_group.columns = ["hit", "ref_protein", "rev_pident", "rev_evalue", "rev_bitscore"]
    rev_group["ref_group"] = group
    rev_tables.append(rev_group)

rev = pd.concat(rev_tables, ignore_index=True)
rev["ref_protein"] = rev["ref_protein"].map(bare_id)
rev.head()

,hit,ref_protein,rev_pident,rev_evalue,rev_bitscore,ref_group
0,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,SCO2401,100.0,5.050000e-295,794.0,SCO
1,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,SCO3475,37.8,3.030000e-70,223.0,SCO
2,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,SCO7629,32.5,6.130000e-56,187.0,SCO
3,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,SCO3480,28.3,1.340000e-32,123.0,SCO
4,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,SCO7570,30.2,8.420000e-27,107.0,SCO


In [17]:
# Sanity check against the file the script already wrote from the same data
rev_combined_on_disk = pd.read_csv(OUTPUT_DIR / "reverse_hits_combined.tsv", sep="\t")
pd.testing.assert_frame_equal(
    rev.reset_index(drop=True), rev_combined_on_disk.reset_index(drop=True)
)
print("rev matches reverse_hits_combined.tsv")

rev matches reverse_hits_combined.tsv


## Step 4 - reciprocal check

Same logic as the script: best reverse hit(s) per (subject, reference organism), then flag a forward hit as reciprocal if the query's true reference id is among them.

In [ ]:
# collapse in-genome paralogs: keep only the top-scoring forward hit per (qseqid, genome)
if GENOME_SEP:
    fwd_hits["genome"] = fwd_hits.sseqid.str.split(GENOME_SEP, regex=False).str[0]

    # best forward bitscore per query, per genome
    top_fwd_bitscore = fwd_hits.groupby(["qseqid", "genome"])["bitscore"].transform("max")

    # only the best hit(s) survive - true orthologs candidates, one per genome per query
    fwd_hits_best = fwd_hits[fwd_hits.bitscore == top_fwd_bitscore]
    fwd_hits_best


,qseqid,sseqid,pident,length,qlen,slen,qstart,qend,sstart,send,evalue,bitscore,full_sseq,query_file,ref_group,true_ref_id,reverse_best_hit,reverse_best_bitscore,reciprocal,genome
0,SCO2401,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,99.7,388,388,388,1,388,1,388,1.170000e-291,791,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,794.0,True,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...
1,SCO2401,Streptomyces_violaceoruber|R2E43_RS26250|mande...,99.7,388,388,388,1,388,1,388,1.170000e-291,791,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,794.0,True,Streptomyces_violaceoruber|R2E43_RS26250|mande...
2,SCO2401,Streptomyces_anthocyanicus|OIE72_RS26120|mande...,99.5,388,388,388,1,388,1,388,3.360000e-291,790,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,793.0,True,Streptomyces_anthocyanicus|OIE72_RS26120|mande...
3,SCO2401,Streptomyces_rubrogriseus|Sru02f_RS06985|mande...,99.0,388,388,388,1,388,1,388,2.760000e-290,788,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,790.0,True,Streptomyces_rubrogriseus|Sru02f_RS06985|mande...
4,SCO2401,Streptomyces_coelicoflavus|OHA99_RS25815|mande...,98.7,388,388,388,1,388,1,388,1.600000e-289,786,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,787.0,True,Streptomyces_coelicoflavus|OHA99_RS25815|mande...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843,vnz_33185-araD,Streptomyces_canus|OHT68_RS00520|L-ribulose-5-...,77.5,236,237,241,1,235,1,236,8.440000e-128,365,MTAHVRDGLRQEVLDANLAIPQVGLATLTWGNVSGVDREAGVFVIK...,vnz_33185,VNZ,vnz_RS33520,vnz_RS33520,371.0,True,Streptomyces_canus|OHT68_RS00520|L-ribulose-5-...
844,vnz_33185-araD,Streptomyces_cynarae|N8I84_RS40475|L-ribulose-...,77.9,235,237,240,1,234,1,235,3.310000e-127,363,MTTAVPESLRQEVLEANLAIPQVGLATLTWGNVSGVDRKSGVFVIK...,vnz_33185,VNZ,vnz_RS33520,vnz_RS33520,370.0,True,Streptomyces_cynarae|N8I84_RS40475|L-ribulose-...
845,vnz_33185-araD,Streptomyces_canus|OHT68_RS02545|L-ribulose-5-...,77.4,235,237,241,1,234,1,235,9.820000e-127,362,MTTTVPESLRREVLDANLAIPQVGLATLTWGNVSGVDREAGVFVIK...,vnz_33185,VNZ,vnz_RS33520,vnz_RS33520,370.0,True,Streptomyces_canus|OHT68_RS02545|L-ribulose-5-...
846,vnz_33185-araD,Streptomyces_prunicolor|OG588_RS46515|L-ribulo...,77.6,237,237,239,1,236,1,237,2.620000e-126,361,MTARVRDGLRQEVLDANLTIPRVGLATLTWGNVSGVDRAAGVFVIK...,vnz_33185,VNZ,vnz_RS33520,vnz_RS33520,368.0,True,Streptomyces_prunicolor|OG588_RS46515|L-ribulo...


In [ ]:
# Step 4, rewrite (v2) - building this up line by line

# max reverse bitscore per (hit, ref_group)
max_bitscore = rev.groupby(["hit", "ref_group"])["rev_bitscore"].transform("max")

# keep only the top-scoring row(s) per group (ties included)
best_rev = rev[rev.rev_bitscore == max_bitscore]

# left-merge so every forward hit is kept; a match means true_ref_id was among the top reverse hits for that sseqid, within its own ref_group
reciprocal_check_df = fwd_hits.merge(
    best_rev[["hit", "ref_group", "ref_protein", "rev_bitscore"]],
    left_on=["true_ref_id", "sseqid", "ref_group"],
    right_on=["ref_protein", "hit", "ref_group"],
    how="left",
).drop(columns=["ref_protein", "hit"])  # only rev_bitscore is kept from best_rev

reciprocal_check_df

,qseqid,sseqid,pident,length,qlen,slen,qstart,qend,sstart,send,...,bitscore,full_sseq,query_file,ref_group,true_ref_id,reverse_best_hit,reverse_best_bitscore,reciprocal,genome,rev_bitscore
0,SCO2401,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,99.7,388,388,388,1,388,1,388,...,791,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,794.0,True,Streptomyces_coelicolor_A3(2)|SCO2401|dehydrat...,794.0
1,SCO2401,Streptomyces_violaceoruber|R2E43_RS26250|mande...,99.7,388,388,388,1,388,1,388,...,791,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,794.0,True,Streptomyces_violaceoruber|R2E43_RS26250|mande...,794.0
2,SCO2401,Streptomyces_anthocyanicus|OIE72_RS26120|mande...,99.5,388,388,388,1,388,1,388,...,790,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,793.0,True,Streptomyces_anthocyanicus|OIE72_RS26120|mande...,793.0
3,SCO2401,Streptomyces_rubrogriseus|Sru02f_RS06985|mande...,99.0,388,388,388,1,388,1,388,...,788,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,790.0,True,Streptomyces_rubrogriseus|Sru02f_RS06985|mande...,790.0
4,SCO2401,Streptomyces_coelicoflavus|OHA99_RS25815|mande...,98.7,388,388,388,1,388,1,388,...,786,MRITGISTHVVGTPWRNLTYVQVHTDEGLTGVGETRMLGHTDALLG...,SCO2401,SCO,SCO2401,SCO2401,787.0,True,Streptomyces_coelicoflavus|OHA99_RS25815|mande...,787.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843,vnz_33185-araD,Streptomyces_canus|OHT68_RS00520|L-ribulose-5-...,77.5,236,237,241,1,235,1,236,...,365,MTAHVRDGLRQEVLDANLAIPQVGLATLTWGNVSGVDREAGVFVIK...,vnz_33185,VNZ,vnz_RS33520,vnz_RS33520,371.0,True,Streptomyces_canus|OHT68_RS00520|L-ribulose-5-...,371.0
844,vnz_33185-araD,Streptomyces_cynarae|N8I84_RS40475|L-ribulose-...,77.9,235,237,240,1,234,1,235,...,363,MTTAVPESLRQEVLEANLAIPQVGLATLTWGNVSGVDRKSGVFVIK...,vnz_33185,VNZ,vnz_RS33520,vnz_RS33520,370.0,True,Streptomyces_cynarae|N8I84_RS40475|L-ribulose-...,370.0
845,vnz_33185-araD,Streptomyces_canus|OHT68_RS02545|L-ribulose-5-...,77.4,235,237,241,1,234,1,235,...,362,MTTTVPESLRREVLDANLAIPQVGLATLTWGNVSGVDREAGVFVIK...,vnz_33185,VNZ,vnz_RS33520,vnz_RS33520,370.0,True,Streptomyces_canus|OHT68_RS02545|L-ribulose-5-...,370.0
846,vnz_33185-araD,Streptomyces_prunicolor|OG588_RS46515|L-ribulo...,77.6,237,237,239,1,236,1,237,...,361,MTARVRDGLRQEVLDANLTIPRVGLATLTWGNVSGVDRAAGVFVIK...,vnz_33185,VNZ,vnz_RS33520,vnz_RS33520,368.0,True,Streptomyces_prunicolor|OG588_RS46515|L-ribulo...,368.0


In [ ]:
# Keep rev hits not already accounted for in reciprocal_check_df's genome column
possible_paralogs = rev[~rev.hit.isin(reciprocal_check_df.genome)]
possible_paralogs

In [ ]:
# Sanity check against the files the script already wrote from the same data
out_cols = [c for c in fwd_hits.columns if c != "full_sseq"]
flagged_on_disk = pd.read_csv(OUTPUT_DIR / "all_forward_hits_flagged.tsv", sep="\t")
pd.testing.assert_frame_equal(
    fwd_hits[out_cols].reset_index(drop=True), flagged_on_disk.reset_index(drop=True)
)
print("fwd_hits matches all_forward_hits_flagged.tsv")

fwd_hits matches all_forward_hits_flagged.tsv


## Manual inspection

Everything below is for poking at specific results - filter/sort as needed.

In [ ]:
reciprocal_hits = fwd_hits.loc[fwd_hits.reciprocal, out_cols]
reciprocal_hits

In [ ]:
possible_paralogs = pd.read_csv(OUTPUT_DIR / "possible_paralogs.tsv", sep="\t")
possible_paralogs

In [ ]:
summary_per_query = pd.read_csv(OUTPUT_DIR / "summary_per_query.tsv", sep="\t")
summary_per_query

In [ ]:
# Drill into a single query, e.g. all its forward hits with reciprocal flag
QUERY = "SCO2401"
fwd_hits.loc[fwd_hits.qseqid == QUERY, out_cols].sort_values("bitscore", ascending=False)